In [1]:
import pandas as pd
import numpy as np

DATA_PATH = "../ingestion/data/processed/district_weather_soil_daily.csv"
df = pd.read_csv(DATA_PATH)

df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["district", "date"]).reset_index(drop=True)

df.head()


,date,district,rainfall_mm,temperature_2m_mean,relative_humidity_2m_mean,surface_pressure_mean,wind_speed_10m_mean,cloud_cover_mean,soil_moisture,soil_moisture_source,soil_source
0,2015-01-01,Chengalpattu,0.0,24.9,87,1003.3,5.6,38,0.29004,simulated,simulated
1,2015-01-02,Chengalpattu,1.3,25.2,88,1005.7,8.0,38,0.28776,simulated,simulated
2,2015-01-03,Chengalpattu,1.2,25.2,89,1006.9,6.4,52,0.28488,simulated,simulated
3,2015-01-04,Chengalpattu,0.4,25.1,88,1006.9,5.3,35,0.27724,simulated,simulated
4,2015-01-05,Chengalpattu,0.2,24.8,83,1006.6,7.9,20,0.26852,simulated,simulated


In [2]:
for window in [3, 7, 14]:
    df[f"rain_{window}d"] = (
        df.groupby("district")["rainfall_mm"]
          .rolling(window)
          .sum()
          .reset_index(level=0, drop=True)
    )

In [3]:
df.head()

,date,district,rainfall_mm,temperature_2m_mean,relative_humidity_2m_mean,surface_pressure_mean,wind_speed_10m_mean,cloud_cover_mean,soil_moisture,soil_moisture_source,soil_source,rain_3d,rain_7d,rain_14d
0,2015-01-01,Chengalpattu,0.0,24.9,87,1003.3,5.6,38,0.29004,simulated,simulated,NaN,NaN,NaN
1,2015-01-02,Chengalpattu,1.3,25.2,88,1005.7,8.0,38,0.28776,simulated,simulated,NaN,NaN,NaN
2,2015-01-03,Chengalpattu,1.2,25.2,89,1006.9,6.4,52,0.28488,simulated,simulated,2.5,NaN,NaN
3,2015-01-04,Chengalpattu,0.4,25.1,88,1006.9,5.3,35,0.27724,simulated,simulated,2.9,NaN,NaN
4,2015-01-05,Chengalpattu,0.2,24.8,83,1006.6,7.9,20,0.26852,simulated,simulated,1.8,NaN,NaN


In [4]:
df[["rainfall_mm", "rain_3d", "rain_7d", "rain_14d"]].head(10)


,rainfall_mm,rain_3d,rain_7d,rain_14d
0,0.0,NaN,NaN,NaN
1,1.3,NaN,NaN,NaN
2,1.2,2.5,NaN,NaN
3,0.4,2.9,NaN,NaN
4,0.2,1.8,NaN,NaN
5,0.3,0.9,NaN,NaN
6,0.0,0.5,3.4,NaN
7,0.7,1.0,4.1,NaN
8,0.8,1.5,3.6,NaN
9,0.0,1.5,2.4,NaN


In [5]:
for window in [7, 14]:
    df[f"soil_{window}d_avg"] = (
        df.groupby("district")["soil_moisture"]
          .rolling(window)
          .mean()
          .reset_index(level=0, drop=True)
    )


In [15]:
df[["soil_moisture", "soil_7d_avg", "soil_14d_avg"]].head(10)

,soil_moisture,soil_7d_avg,soil_14d_avg
0,0.19420,0.221200,0.247726
1,0.18488,0.212600,0.240214
2,0.17940,0.203897,0.232474
3,0.16964,0.195120,0.224243
4,0.16016,0.186309,0.215880
5,0.15072,0.177474,0.207466
6,0.14128,0.168611,0.198949
7,0.13192,0.159714,0.190457
8,0.12672,0.151406,0.182003
9,0.12420,0.143520,0.173709


In [7]:
df["wetness_index"] = (
    df["rain_7d"] * df["soil_7d_avg"]
)


In [8]:
df["rain_intensity"] = df["rainfall_mm"] / (df["rain_3d"] + 1e-3)


In [9]:
df["month"] = df["date"].dt.month
df["is_monsoon"] = df["month"].isin([6,7,8,9,10,11,12]).astype(int)


In [16]:
df.head(15)


,date,district,rainfall_mm,temperature_2m_mean,relative_humidity_2m_mean,surface_pressure_mean,wind_speed_10m_mean,cloud_cover_mean,soil_moisture,soil_moisture_source,soil_source,rain_3d,rain_7d,rain_14d,soil_7d_avg,soil_14d_avg,wetness_index,rain_intensity,month,is_monsoon
0,2015-01-14,Chengalpattu,0.0,22.8,71,1007.3,9.4,13,0.19420,simulated,simulated,0.0,1.5,4.9,0.221200,0.247726,0.331800,0.000000,1,0
1,2015-01-15,Chengalpattu,0.0,23.3,77,1007.4,8.7,27,0.18488,simulated,simulated,0.0,0.8,4.9,0.212600,0.240214,0.170080,0.000000,1,0
2,2015-01-16,Chengalpattu,0.7,24.2,80,1007.7,10.8,29,0.17940,simulated,simulated,0.7,0.7,4.3,0.203897,0.232474,0.142728,0.998573,1,0
3,2015-01-17,Chengalpattu,0.0,24.4,74,1008.2,10.3,57,0.16964,simulated,simulated,0.7,0.7,3.1,0.195120,0.224243,0.136584,0.000000,1,0
4,2015-01-18,Chengalpattu,0.0,23.7,71,1008.9,8.7,73,0.16016,simulated,simulated,0.7,0.7,2.7,0.186309,0.215880,0.130416,0.000000,1,0
5,2015-01-19,Chengalpattu,0.0,23.6,71,1008.6,9.4,15,0.15072,simulated,simulated,0.0,0.7,2.5,0.177474,0.207466,0.124232,0.000000,1,0
6,2015-01-20,Chengalpattu,0.0,23.6,69,1010.1,9.9,5,0.14128,simulated,simulated,0.0,0.7,2.2,0.168611,0.198949,0.118028,0.000000,1,0
7,2015-01-21,Chengalpattu,0.0,23.4,68,1011.4,10.6,5,0.13192,simulated,simulated,0.0,0.7,2.2,0.159714,0.190457,0.111800,0.000000,1,0
8,2015-01-22,Chengalpattu,0.7,23.5,79,1011.0,10.5,34,0.12672,simulated,simulated,0.7,1.4,2.2,0.151406,0.182003,0.211968,0.998573,1,0
9,2015-01-23,Chengalpattu,1.2,24.3,82,1008.7,8.7,76,0.12420,simulated,simulated,1.9,1.9,2.6,0.143520,0.173709,0.272688,0.631247,1,0


In [12]:
df = df.dropna().reset_index(drop=True)


In [14]:
(df.head(15))

,date,district,rainfall_mm,temperature_2m_mean,relative_humidity_2m_mean,surface_pressure_mean,wind_speed_10m_mean,cloud_cover_mean,soil_moisture,soil_moisture_source,soil_source,rain_3d,rain_7d,rain_14d,soil_7d_avg,soil_14d_avg,wetness_index,rain_intensity,month,is_monsoon
0,2015-01-14,Chengalpattu,0.0,22.8,71,1007.3,9.4,13,0.19420,simulated,simulated,0.0,1.5,4.9,0.221200,0.247726,0.331800,0.000000,1,0
1,2015-01-15,Chengalpattu,0.0,23.3,77,1007.4,8.7,27,0.18488,simulated,simulated,0.0,0.8,4.9,0.212600,0.240214,0.170080,0.000000,1,0
2,2015-01-16,Chengalpattu,0.7,24.2,80,1007.7,10.8,29,0.17940,simulated,simulated,0.7,0.7,4.3,0.203897,0.232474,0.142728,0.998573,1,0
3,2015-01-17,Chengalpattu,0.0,24.4,74,1008.2,10.3,57,0.16964,simulated,simulated,0.7,0.7,3.1,0.195120,0.224243,0.136584,0.000000,1,0
4,2015-01-18,Chengalpattu,0.0,23.7,71,1008.9,8.7,73,0.16016,simulated,simulated,0.7,0.7,2.7,0.186309,0.215880,0.130416,0.000000,1,0
5,2015-01-19,Chengalpattu,0.0,23.6,71,1008.6,9.4,15,0.15072,simulated,simulated,0.0,0.7,2.5,0.177474,0.207466,0.124232,0.000000,1,0
6,2015-01-20,Chengalpattu,0.0,23.6,69,1010.1,9.9,5,0.14128,simulated,simulated,0.0,0.7,2.2,0.168611,0.198949,0.118028,0.000000,1,0
7,2015-01-21,Chengalpattu,0.0,23.4,68,1011.4,10.6,5,0.13192,simulated,simulated,0.0,0.7,2.2,0.159714,0.190457,0.111800,0.000000,1,0
8,2015-01-22,Chengalpattu,0.7,23.5,79,1011.0,10.5,34,0.12672,simulated,simulated,0.7,1.4,2.2,0.151406,0.182003,0.211968,0.998573,1,0
9,2015-01-23,Chengalpattu,1.2,24.3,82,1008.7,8.7,76,0.12420,simulated,simulated,1.9,1.9,2.6,0.143520,0.173709,0.272688,0.631247,1,0


In [22]:
df =df.drop(columns=['soil_moisture_source','soil_source'])

In [23]:
df.head()

,date,district,rainfall_mm,temperature_2m_mean,relative_humidity_2m_mean,surface_pressure_mean,wind_speed_10m_mean,cloud_cover_mean,soil_moisture,rain_3d,rain_7d,rain_14d,soil_7d_avg,soil_14d_avg,wetness_index,rain_intensity,month,is_monsoon
0,2015-01-14,Chengalpattu,0.0,22.8,71,1007.3,9.4,13,0.19420,0.0,1.5,4.9,0.221200,0.247726,0.331800,0.000000,1,0
1,2015-01-15,Chengalpattu,0.0,23.3,77,1007.4,8.7,27,0.18488,0.0,0.8,4.9,0.212600,0.240214,0.170080,0.000000,1,0
2,2015-01-16,Chengalpattu,0.7,24.2,80,1007.7,10.8,29,0.17940,0.7,0.7,4.3,0.203897,0.232474,0.142728,0.998573,1,0
3,2015-01-17,Chengalpattu,0.0,24.4,74,1008.2,10.3,57,0.16964,0.7,0.7,3.1,0.195120,0.224243,0.136584,0.000000,1,0
4,2015-01-18,Chengalpattu,0.0,23.7,71,1008.9,8.7,73,0.16016,0.7,0.7,2.7,0.186309,0.215880,0.130416,0.000000,1,0


In [26]:
df.sample(15)

,date,district,rainfall_mm,temperature_2m_mean,relative_humidity_2m_mean,surface_pressure_mean,wind_speed_10m_mean,cloud_cover_mean,soil_moisture,rain_3d,rain_7d,rain_14d,soil_7d_avg,soil_14d_avg,wetness_index,rain_intensity,month,is_monsoon
19840,2019-07-12,Tiruvallur,0.3,30.9,63,1002.5,9.8,49,0.00000,2.0,2.3,3.5,0.011097,0.050614,0.025523,0.149925,7,1
10433,2023-09-02,Cuddalore,3.3,29.5,77,1005.9,12.7,38,1.00000,12.6,45.3,62.9,0.994143,0.979180,45.034671,0.261884,9,1
18542,2015-12-22,Tiruvallur,0.0,24.1,82,1010.1,5.9,22,0.88456,0.0,0.6,14.4,0.912063,0.943823,0.547238,0.000000,12,1
13914,2023-03-27,Kancheepuram,0.2,29.6,73,998.8,8.7,35,0.42876,1.7,5.5,47.0,0.451143,0.398206,2.481286,0.117578,3,0
19526,2018-09-01,Tiruvallur,0.6,28.7,72,1001.8,9.8,88,0.19916,12.8,20.1,28.6,0.177177,0.171017,3.561261,0.046871,9,1
4791,2018-03-10,Chennai,0.0,26.6,79,1010.0,8.3,58,0.25772,0.0,0.0,0.0,0.288880,0.324517,0.000000,0.000000,3,0
1197,2018-04-25,Chengalpattu,0.0,30.1,73,1001.6,12.9,39,0.00000,1.9,2.0,11.5,0.000000,0.012746,0.000000,0.000000,4,0
3609,2024-12-01,Chengalpattu,18.0,25.5,91,994.8,23.7,100,1.00000,219.4,252.6,256.1,0.989937,0.975649,250.058122,0.082042,12,1
13114,2021-01-16,Kancheepuram,0.0,23.8,82,1000.6,8.6,55,0.97768,3.2,7.9,56.7,0.991274,0.995077,7.831067,0.000000,1,0
683,2016-11-27,Chengalpattu,0.0,24.5,72,1005.7,8.8,28,0.69632,0.0,0.0,3.9,0.726571,0.760649,0.000000,0.000000,11,1


In [25]:
df.to_csv("../ingestion/data/processed/district_weather_soil_features_clean.csv", index=False)

In [28]:
df.columns.tolist()

['date',
 'district',
 'rainfall_mm',
 'temperature_2m_mean',
 'relative_humidity_2m_mean',
 'surface_pressure_mean',
 'wind_speed_10m_mean',
 'cloud_cover_mean',
 'soil_moisture',
 'rain_3d',
 'rain_7d',
 'rain_14d',
 'soil_7d_avg',
 'soil_14d_avg',
 'wetness_index',
 'rain_intensity',
 'month',
 'is_monsoon']